# Bronze - Crew Data

## Setup Environment and Prepare data 

In [2]:
from letterboxd_data_pipeline.load_data import load_files_from_layer

bronze_df = load_files_from_layer(layer="bronze", file_list=["crew"])
bronze_crew_df = bronze_df["crew"]

Start loading data...
Loading: data/bronze/crew
Done!
Finish loading.


## Simple Explore

In [3]:
bronze_crew_df.describe(include="all")

,id,role,name
count,4.720183e+06,4720183,4720182
unique,NaN,29,1147402
top,NaN,Director,Louis Lumière
freq,NaN,900753,1176
mean,1.306020e+06,NaN,NaN
std,2.692652e+05,NaN,NaN
min,1.000001e+06,NaN,NaN
25%,1.065264e+06,NaN,NaN
50%,1.230888e+06,NaN,NaN
75%,1.506076e+06,NaN,NaN


In [4]:
bronze_crew_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4720183 entries, 0 to 4720182
Data columns (total 3 columns):
 #   Column  Dtype 
---  ------  ----- 
 0   id      int64 
 1   role    object
 2   name    object
dtypes: int64(1), object(2)
memory usage: 108.0+ MB


## Distinct crew roles count

In [5]:
bronze_crew_df.groupby("role").agg(count=("role","count")).sort_values(by="count",ascending=False)

,count
role,
Director,900753
Writer,677156
Producer,566464
Cinematography,351618
Editor,326088
Sound,298333
Composer,259650
Executive producer,202770
Stunts,114335


## Crew member that multi hat per movie

In [6]:
source_df = bronze_crew_df.copy(True)

# Group the dataframe
grouped_source_df = source_df.groupby(by=["id", "name"])
# Filter "name" count < 2
multi_hat_crew_per_movie = grouped_source_df.filter(
    lambda x: x["name"].value_counts() > 1
)
# Add new column count for name
multi_hat_crew_per_movie["role_count"] = multi_hat_crew_per_movie.groupby(by=["id", "name"])[
    "name"
].transform(len)
# Order columns and sort by "id" and "count"
multi_hat_crew_per_movie = multi_hat_crew_per_movie[["id", "name", "role", "role_count"]].sort_values(
    ["id", "role_count","name"], ascending=[True, False, True]
)

multi_hat_crew_per_movie

,id,name,role,role_count
0,1000001,Greta Gerwig,Director,3
7,1000001,Greta Gerwig,Writer,3
29,1000001,Greta Gerwig,Executive producer,3
109,1000001,Andrew Wyatt,Composer,2
114,1000001,Andrew Wyatt,Songs,2
...,...,...,...,...
4720162,1941542,Axel Loh,Writer,2
4720164,1941548,Mohammad Salehinejad,Director,2
4720166,1941548,Mohammad Salehinejad,Writer,2
4720173,1941596,Marc Ma,Director,2


In [7]:
multi_hat_crew_per_movie.describe().apply(lambda s: s.apply('{0:.5f}'.format))

,id,role_count
count,1209016.00000,1209016.00000
mean,1381793.92738,2.69714
std,271327.35977,1.06970
min,1000001.00000,2.00000
25%,1136526.00000,2.00000
50%,1347535.00000,2.00000
75%,1604809.00000,3.00000
max,1941596.00000,15.00000


## Highest hat per movie record details

In [8]:
max_hat_movie_index = multi_hat_crew_per_movie['role_count'].idxmax()
max_hat_movie_id = multi_hat_crew_per_movie.loc[max_hat_movie_index]["id"]
max_hat_movie_id

np.int64(1522556)

In [9]:
multi_hat_crew_per_movie[multi_hat_crew_per_movie["id"] == max_hat_movie_id]

## this person is a one man show

,id,name,role,role_count
3597965,1522556,Stamatianna,Director,15
3597966,1522556,Stamatianna,Producer,15
3597967,1522556,Stamatianna,Writer,15
3597968,1522556,Stamatianna,Original writer,15
3597969,1522556,Stamatianna,Casting,15
3597970,1522556,Stamatianna,Editor,15
3597971,1522556,Stamatianna,Cinematography,15
3597972,1522556,Stamatianna,Lighting,15
3597973,1522556,Stamatianna,Camera operator,15
3597975,1522556,Stamatianna,Production design,15


## Number of NA

In [10]:
bronze_crew_df.isna().sum()

id      0
role    0
name    1
dtype: int64